In [5]:
# 1. Construct the Neutrino client. Read only the PAT from ./config.json.
import contextlib
import io
import json
from pathlib import Path

import torch

from dss_client import NeutrinoClient, serialize_forward_backward_args

PAT = json.loads(Path("../config.json").read_text(encoding="utf-8"))["pat"]
client = NeutrinoClient.from_pat(
    host="dsa-test.qa6.us-west-2.aws.snowflakecomputing.com",
    pat=PAT,
    database="NEUTRINO_DB",
    schema="PUBLIC",
    endpoint="cortex-training",
    poll_interval=0.5,
    poll_timeout=1800.0,
    verify_ssl=True,
)
del PAT

In [6]:
# 2. Construct an eight-GPU Qwen3-8B SFT job on an existing job's GPUs.
SOURCE_JOB_ID = "5d267f27-682b-4173-8fcc-f515121c82f4"
MULTIPLEX_JOB_ID = f"{SOURCE_JOB_ID}:training:0"

job_body = {
    "sub_job_configs": [
        {
            "job_type": "training",
            "model_name": "Qwen/Qwen3-8B",
            "dtype": "bfloat16",
            "seed": 42,
            "training_config": {
                "model_provider": "huggingface",
                "n_gpus": 8,
                "max_seq_len": 128,
                "train_batch_size": 8,
                "multiplex_job_id": MULTIPLEX_JOB_ID,
                "attn_implementation": "flash_attention_3",
                "optimizer": {
                    "name": "AdamW",
                    "lr": 0.0001,
                    "weight_decay": 0.0,
                    "betas": [0.9, 0.999],
                    "eps": 0.00000001,
                },
                "ds_config": {
                    "train_batch_size": 8,
                    "train_micro_batch_size_per_gpu": 1,
                    "gradient_accumulation_steps": 1,
                    "zero_optimization": {"stage": 2},
                    "bf16": {"enabled": True},
                },
            },
        }
    ]
}

In [7]:
# 3. Submit the job and wait until its workers are running (will take about 3 min).
job_id = client.create_job_from_body(job_body)["job_id"]
print(f"Job {job_id} is starting")
client.wait_for_job(job_id)
print(f"Job {job_id} is running")

Job 6f69fd40-a232-467e-acfb-20ab1c9660ac is starting
Job 6f69fd40-a232-467e-acfb-20ab1c9660ac is running


In [8]:
# 4. Generate five random SFT batches with causal next-token labels.
torch.manual_seed(42)
BATCH_SIZE = 8
SEQ_LEN = 128


def make_random_batch():
    input_ids = torch.randint(0, 1000, (BATCH_SIZE, SEQ_LEN))
    labels = torch.roll(input_ids, shifts=-1, dims=1)
    labels[:, -1] = -100
    return {
        "input_ids": input_ids,
        "position_ids": torch.arange(SEQ_LEN).expand(BATCH_SIZE, -1).contiguous(),
        "labels": labels,
    }


training_data = [make_random_batch() for _ in range(5)]

In [9]:
# 5. Run forward-backward and one optimizer update for each batch.
for step_number, batch in enumerate(training_data, start=1):
    payload = serialize_forward_backward_args(args=(), kwargs=batch)
    with contextlib.redirect_stdout(io.StringIO()):
        request_id = client.forward_backward(job_id, payload)
        result = client.poll_request(job_id, request_id)
        request_id = client.step(job_id, learning_rate=0.0001)
        client.poll_request(job_id, request_id)
    print(f"step={step_number} loss={result['avg_loss']}")

step=1 loss=10.504879236221313
step=2 loss=9.295836210250854
step=3 loss=8.825100779533386
step=4 loss=9.06827962398529
step=5 loss=8.280941843986511


In [10]:
# 6. Tear down the remote job and release its GPUs.
client.cancel_job(job_id)
print(f"Cancelled job {job_id}")
job_id = None

Cancelled job 6f69fd40-a232-467e-acfb-20ab1c9660ac
